# Activity bars with Yahoo OHLCV data

This notebook builds daily activity-based bars using the OHLCV data downloaded from Yahoo Finance.

Generated bar types:

- Time bars
- Count bars as a daily proxy for tick bars
- Volume bars using aggregated universe volume
- Dollar bars using aggregated universe dollar volume


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "util.py").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "util.py").exists())

sys.path.insert(0, str(PROJECT_ROOT / "model" / "preprocessing"))

import pandas as pd

from preprocessing_utils import (
    DATA_OUT,
    PLOTS_DIR,
    build_activity_bars,
    summarize_bars,
    save_bars,
    plot_bar_counts,
    plot_return_distributions,
    plot_bar_durations,
    create_preprocessing_report,
)

TARGET_BARS = int(os.getenv("TARGET_BARS", "1000"))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_OUT:", DATA_OUT)
print("TARGET_BARS:", TARGET_BARS)


## Load OHLCV data

In [ ]:
ohlcv_path = DATA_OUT / "yahoo_ohlcv.parquet"
if not ohlcv_path.exists():
    raise FileNotFoundError("Run 01_yahoo_ohlcv_audit.ipynb first to create data/preprocessing/yahoo_ohlcv.parquet")

ohlcv = pd.read_parquet(ohlcv_path)
print("OHLCV shape:", ohlcv.shape)
display(ohlcv.head())


## Build activity bars

A common bar calendar is built using the aggregated activity of the full asset universe. This keeps the transformed close/return matrices compatible with the multivariate forecasting setup.

In [ ]:
bars = build_activity_bars(ohlcv, target_bars=TARGET_BARS)
summary = summarize_bars(bars)

summary_path = DATA_OUT / "activity_bars_summary.csv"
summary.to_csv(summary_path, index=False)

save_bars(bars, DATA_OUT)

print("Saved:", summary_path)
display(summary)


## Generate plots

In [ ]:
bar_counts_path = PLOTS_DIR / "activity_bars_counts.png"
return_dist_path = PLOTS_DIR / "activity_bars_return_distributions.png"
durations_path = PLOTS_DIR / "activity_bars_durations.png"

plot_bar_counts(summary, bar_counts_path)
plot_return_distributions(bars, return_dist_path)
plot_bar_durations(bars, durations_path)

print("Saved:", bar_counts_path)
print("Saved:", return_dist_path)
print("Saved:", durations_path)


## Returns distribution summary

In [ ]:
return_rows = []
for name, item in bars.items():
    ret = item["returns"]
    per_asset = pd.DataFrame({
        "ticker": ret.columns,
        "bar_type": name,
        "mean_abs_return": ret.abs().mean(axis=0).values,
        "return_std": ret.std(axis=0).values,
        "missing_values": ret.isna().sum(axis=0).values,
    })
    return_rows.append(per_asset)

returns_summary = pd.concat(return_rows, ignore_index=True)
returns_summary_path = DATA_OUT / "returns_distribution_summary.csv"
returns_summary.to_csv(returns_summary_path, index=False)

print("Saved:", returns_summary_path)
display(returns_summary.head(20))


## Create report

In [ ]:
report_path = create_preprocessing_report(summary)
print("Saved:", report_path)
